# CatBoost v5 — Per-Security, Horizon-Sweep

**Third benchmark notebook** in the Norwegian swap rate forecastability study.
Methodologically identical to `model_htboost_v5_clean.ipynb` and
`scripts/model_xgboost_v5_clean.py` in every respect except the model.
Logic reference: XGBoost script (most recently validated pipeline).
Format/structure reference: HTBoost notebook.

**Three documented, known divergences** (all stated in the thesis):

1. **Loss function** — CatBoost uses RMSE (squared error). HTBoost uses Student-t loss
   (`LOSS='t'`). The comparison is NOT a perfect isolation of the smooth-split mechanism.
   XGBoost also uses squared error; see XGBoost script docstring for the identical note.

2. **Bootstrap / subsampling** — CatBoost uses Minimum Variance Sampling (MVS, the CPU
   default) rather than Bernoulli random subsampling. MVS preferentially resamples
   observations with high gradient variance. The `subsample` grid parameter is therefore
   the MVS *data fraction*, not XGBoost-style random subsampling; see column
   `cat_mvs_fraction` in the tuning log. This is a second known divergence, analogous to
   the loss-function difference — we let CatBoost use its native mechanism.

3. **Tree structure** — CatBoost builds symmetric (oblivious) trees: all nodes at a given
   depth use the same feature and threshold. This is architecturally distinct from XGBoost's
   greedy asymmetric trees. `depth` values are NOT directly comparable across models (same
   depth integer ≠ same tree shape or flexibility); this is part of CatBoost's distinct
   character as a third comparison point.

Output schema is byte-identical to both HTBoost and XGBoost on `SHARED_COLS + pca_k +
xm_pca_evr`; model-specific diagnostic columns are `cat_n_trees`, `cat_depth`, `cat_lr`.

## Setup

Import the scientific-Python stack, statistical-test libraries, and CatBoost (no Julia
dependency). All shared modules are imported unchanged from `src/`.

In [ ]:
import re
import os
import sys
import json
import hashlib
import itertools
import pickle
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.metrics import r2_score

# catboost 1.2.10 — pip-installed; not yet in environment.yml (tracked separately).
# Version pinned here for appendix reproducibility.
from catboost import CatBoostRegressor, Pool  # catboost==1.2.10

# ── Repo root resolution (same pattern as the HTBoost notebook) ───────────────
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

# ── Shared modules — imported unchanged; no modifications to any src/ file ────
from src.data.bloomberg import load_data
from src.data.norway import load_norway_raw, print_connectivity_report
from src.features.macro import add_macro_features
from src.features.norway import add_norway_features
from src.features.panel import build_panel
from src.features.pca import fit_transform_xm_pca
from src.evaluation.metrics import (
    compute_metrics_row,
    SHARED_COLS,
    MTC_COLS,
    apply_mtc_family,
    finalize_long_csv,
)
import src.config as config
from src.config import MACHINE_ID
from src.features.taxonomy import bucket_feature, BUCKETS
from src.utils.gitpush import push_results
import matplotlib.pyplot as plt
try:
    from thesis_style import apply_thesis_style, OKABE_ITO
    apply_thesis_style()
except Exception:
    OKABE_ITO = ['#0072B2','#E69F00','#009E73','#D55E00','#CC79A7','#56B4E9','#F0E442','#000000']

warnings.filterwarnings('ignore')
print(f'Imports OK  (catboost 1.2.10, pandas {pd.__version__}, numpy {np.__version__})')
print(f'MACHINE_ID = {MACHINE_ID!r}')
print(f'_ROOT      = {_ROOT!r}')

### Configuration

All run parameters in one place. Nothing here is chosen by looking at an OOS fold.
`SMOKE_MODE = True` (default): runs only NOR_10Y × H=21, writes to `_smoke/` subdirectory.
Set `SMOKE_MODE = False` for the full sweep.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIG — all run parameters live here; nothing is selected on OOS performance
# ════════════════════════════════════════════════════════════════════════════

NOTEBOOK   = 'catboost_v5'      # short provenance label (stamped in every metrics row)
MODEL_KIND = 'per_security'     # same as HTBoost / XGBoost per-security notebooks
IS_POOLED  = False
RUN_TS     = datetime.now(tz=timezone.utc).isoformat()

# SMOKE_MODE: True → NOR_10Y × H=21 only (→ _smoke/); False → full sweep
SMOKE_MODE = True

# Horizon grid — identical to HTBoost / XGBoost
H_GRID      = config.H_GRID       # [1, 5, 21, 63]
H_GRID_LONG = [126, 252]          # 6m, 12m — gated on data length (same rule as HTBoost)

# Walk-forward folds — identical to HTBoost / XGBoost (from shared config)
WF_FOLDS   = config.WF_FOLDS
FOLD_NAMES = config.FOLD_NAMES

# Universe thresholds — identical to HTBoost / XGBoost
UNIVERSE_MIN_OBS = 500
MIN_TRAIN_OBS    = config.MIN_TRAIN_OBS   # 252
MIN_TEST_OBS     = config.MIN_TEST_OBS    # 20
TARGET_LAGS      = 5                       # AR lags of chg_1d — identical to HTBoost / XGBoost

# PCA compression of the cross-market xm_* block — identical to HTBoost / XGBoost
XM_PCA_ENABLE = config.XM_PCA_ENABLE
XM_PCA_VAR    = config.XM_PCA_VAR    # 0.95
XM_PCA_KMAX   = config.XM_PCA_KMAX   # 12

# Compressed fingerprint of the feature specification; used in config_hash
FEAT_SPEC = f'pca_var{XM_PCA_VAR}_kmax{XM_PCA_KMAX}_tl{TARGET_LAGS}'

# Output directory — SEPARATE from HTBoost (v5_nor/) and XGBoost (xgb_v5_nor/)
OUT_DIR   = os.path.join(_ROOT, 'results', 'catboost_v5_nor')
SMOKE_DIR = os.path.join(OUT_DIR, '_smoke')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
# Block-CV / LORO folds + embargo (shared config) — for the non-causal learnability family.
BLOCK_CV_FOLDS = config.BLOCK_CV_FOLDS
EMBARGO_FOR_H  = config.EMBARGO_FOR_H
# CatBoost per-feature importance types (analog of XGBoost gain/cover/weight). SHAP added
# separately via native ShapValues. PredictionValuesChange = default (XGBoost-gain-like).
IMP_TYPES = ('PredictionValuesChange', 'LossFunctionChange')
# Run switches (parity with the XGBoost notebook).
RUN_WF = True; RUN_BLOCKCV = True; RUN_LORO = True
RUN_PUSH = True   # auto-push per horizon via src.utils.gitpush (HTBoost/XGBoost parity)

# Norway feature cache (cache-first, same as XGBoost script — live=False)
NOR_CACHE = os.path.join(_ROOT, 'data', 'cache', 'norway_raw_features.csv')

# Smoke test defaults
SMOKE_SEC = 'NOR_10Y'
SMOKE_H   = 21

META_COLS = {'date', 'security', 'y', 'level'}

# ════════════════════════════════════════════════════════════════════════════
# CatBoost hyperparameter tuning
# Pre-committed grid; no OOS fold ever influences these values.
#
# KNOWN DIVERGENCE 1 (loss): CatBoost uses RMSE. HTBoost uses Student-t.
# KNOWN DIVERGENCE 2 (bootstrap): CatBoost uses MVS (Minimum Variance Sampling,
#   the CPU default). XGBoost uses Bernoulli random subsampling. The `subsample`
#   parameter here is the MVS *data fraction* — it is NOT equivalent to XGBoost's
#   `subsample` (Bernoulli). Labelled `cat_mvs_fraction` in the tuning log and
#   any reported hyperparameter table to prevent apples-to-oranges comparison.
# KNOWN DIVERGENCE 3 (tree structure): CatBoost builds symmetric (oblivious) trees.
#   `depth` [4, 6] is NOT directly comparable to XGBoost `max_depth` [3, 5]:
#   at equal depth integers, CatBoost's symmetric constraint produces different
#   tree shapes and different numbers of effective decision rules. This is part of
#   CatBoost's distinct character; we do not suppress it to force parity.
# ════════════════════════════════════════════════════════════════════════════

LOSS_FUNCTION  = 'RMSE'
N_TREES_MAX    = 500     # maximum boosting rounds (early stopping applied in inner CV)
EARLY_STOP_RND = 20      # patience: rounds without improvement before stopping
INNER_VAL_FRAC = 0.20    # time-ordered inner validation: last 20% of training window

# Pre-committed hyperparameter grid (tuned within training window only).
# 6 parameters × same factor counts as XGBoost → 3×2×2×2×3×2 = 144 combinations.
CAT_PARAM_GRID = {
    'learning_rate':      [0.01, 0.05, 0.10],
    # CatBoost oblivious-tree depth: all splits at a level share one feature/threshold.
    # Range [4, 6] is appropriate for this architecture; not equivalent to XGBoost depth.
    'depth':              [4, 6],
    # MVS data fraction (NOT Bernoulli subsampling — see KNOWN DIVERGENCE 2 above).
    # Logged as cat_mvs_fraction in the tuning CSV to prevent misreading.
    'subsample':          [0.8, 1.0],
    # rsm: random subspace method — fraction of features considered per split.
    # Direct analog of XGBoost colsample_bytree.
    'rsm':                [0.8, 1.0],
    # L2 leaf regularization — direct analog of XGBoost reg_lambda.
    'l2_leaf_reg':        [1.0, 5.0, 10.0],
    # Minimum number of training samples required in a leaf.
    # Closest analog of XGBoost min_child_weight (integer count vs hessian sum).
    'min_data_in_leaf':   [1, 5],
}
# Grid size: 3 × 2 × 2 × 2 × 3 × 2 = 144 combinations

print(f'SMOKE_MODE={SMOKE_MODE}  NOTEBOOK={NOTEBOOK!r}  MODEL_KIND={MODEL_KIND!r}')
print(f'H_GRID={H_GRID}  (+long {H_GRID_LONG} if data supports)')
print(f'LOSS_FUNCTION={LOSS_FUNCTION!r}  N_TREES_MAX={N_TREES_MAX}  '
      f'EARLY_STOP_RND={EARLY_STOP_RND}  INNER_VAL_FRAC={INNER_VAL_FRAC}')
print(f'PCA: enable={XM_PCA_ENABLE}  var≥{XM_PCA_VAR}  kmax={XM_PCA_KMAX}')
print(f'OUT_DIR={OUT_DIR!r}')
_n_combos = len(list(itertools.product(*CAT_PARAM_GRID.values())))
print(f'CatBoost param grid ({_n_combos} combinations):')
for _k, _v in CAT_PARAM_GRID.items():
    _note = '← MVS data fraction, NOT Bernoulli subsample' if _k == 'subsample' else ''
    print(f'  {_k:<20s}: {_v}  {_note}')

### Helper utilities

`_config_hash`, `_score`, `_prepare_x_cat` (column-name sanitization + ±inf→NaN),
`_done_secs` / `_append_csv` (checkpoint helpers identical to XGBoost script),
and file-path helpers (analogous to `V5_WF_CSV` etc. in the HTBoost notebook).

In [ ]:
def _config_hash(cfg, extra=''):
    """Stable 12-char MD5 fingerprint of a config dict. Identical logic to HTBoost notebook."""
    payload = (json.dumps({k: cfg.get(k) for k in sorted(cfg)},
                           sort_keys=True, default=str)
               + '|' + str(extra))
    return hashlib.md5(payload.encode()).hexdigest()[:12]


def _score(y, yhat):
    """DirAcc / R² / n_obs helper for diagnostic printing."""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = np.isfinite(y) & np.isfinite(yhat)
    if m.sum() < 5:
        return None
    return {'dir_acc': float(np.mean(np.sign(y[m]) == np.sign(yhat[m]))),
            'r2':      float(r2_score(y[m], yhat[m])),
            'n_obs':   int(m.sum())}


def _prepare_x_cat(df):
    """Sanitize column names and replace ±inf with NaN.
    CatBoost handles NaN natively for tree-building (no fillna needed).
    The xmpca_* block is already zero-filled by fit_transform_xm_pca."""
    df = df.copy()
    rmap = {c: re.sub(r'[^a-zA-Z0-9_]', '_', str(c)).strip('_') for c in df.columns}
    df = df.rename(columns=rmap)
    # Deduplicate any name collisions created by sanitization (rare with these features)
    seen, new_cols = {}, []
    for c in df.columns:
        if c in seen:
            seen[c] += 1
            new_cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0
            new_cols.append(c)
    df.columns = new_cols
    return df.replace([np.inf, -np.inf], np.nan).astype(np.float64)


def _done_secs(csv_path):
    """Securities whose full fold-batch is already written to csv_path."""
    if not os.path.exists(csv_path):
        return set()
    try:
        return set(pd.read_csv(csv_path, usecols=['security'])['security'].astype(str))
    except Exception as e:
        print(f'  [resume] could not read {csv_path} ({repr(e)[:50]}) — treating as empty')
        return set()


def _append_csv(df, path):
    """Append DataFrame rows to a CSV, writing header on first write. Flush + fsync."""
    write_header = not os.path.exists(path)
    with open(path, 'a', newline='') as fh:
        df.to_csv(fh, header=write_header, index=False)
        fh.flush()
        os.fsync(fh.fileno())


# File-path helpers (analogous to V5_WF_CSV etc. in the HTBoost notebook)
def _wf_csv(h, out_dir):     return os.path.join(out_dir, f'cat_wf_H{h}__{MACHINE_ID}.csv')
def _pred_csv(h, out_dir):   return os.path.join(out_dir, f'cat_wf_preds_H{h}__{MACHINE_ID}.csv')
def _imp_csv(h, out_dir):    return os.path.join(out_dir, f'cat_wf_imps_H{h}__{MACHINE_ID}.csv')
def _imp_pkl(h, out_dir):    return os.path.join(out_dir, f'cat_wf_imps_H{h}__{MACHINE_ID}.pkl')
def _tuning_csv(h, out_dir): return os.path.join(out_dir, f'cat_tuning_log_H{h}__{MACHINE_ID}.csv')
def _blk_csv(h, out_dir):      return os.path.join(out_dir, f'cat_blockcv_H{h}__{MACHINE_ID}.csv')
def _blk_pred_csv(h, out_dir): return os.path.join(out_dir, f'cat_blockcv_preds_H{h}__{MACHINE_ID}.csv')
def _blk_imp_csv(h, out_dir):  return os.path.join(out_dir, f'cat_blockcv_imps_H{h}__{MACHINE_ID}.csv')
def _evalhist_csv(h, out_dir): return os.path.join(out_dir, f'cat_eval_history_H{h}__{MACHINE_ID}.csv')
def _shap_csv(h, out_dir):     return os.path.join(out_dir, f'cat_shap_buckets_H{h}__{MACHINE_ID}.csv')

def _maybe_push(paths, msg):
    paths = [p for p in paths if p and os.path.exists(p)]
    if RUN_PUSH and paths:
        try: push_results(paths, msg)
        except Exception as e: print(f'  [push] skipped - {repr(e)[:80]}')

def _load_pkl(p):
    if os.path.exists(p):
        try:
            with open(p, 'rb') as f: return pickle.load(f)
        except Exception: pass
    return []

def _dump_pkl(p, obj):
    with open(p, 'wb') as f: pickle.dump(obj, f); f.flush(); os.fsync(f.fileno())

print('Helper utilities and file-path helpers defined.')

### Load data and define the universe

Replicates the data-preparation steps of the HTBoost notebook and XGBoost script in order:
`load_data()` → `load_norway_raw()` (cache-first) → `add_norway_features()` → `add_macro_features()`.
Universe: NOR swap columns with ≥ `UNIVERSE_MIN_OBS` non-NaN observations.

In [ ]:
def load_and_augment_data():
    """Load Bloomberg data and augment with macro + Norway features.
    Steps are byte-identical to the XGBoost script's load_and_augment_data().
    """
    print('Loading Bloomberg data ...')
    df_raw = load_data()
    print(f'  df_raw shape: {df_raw.shape}   '
          f'({df_raw.index.min().date()} → {df_raw.index.max().date()})')

    print('Loading Norway data (cache-first) ...')
    start_str = df_raw.index.min().strftime('%Y-%m-%d')
    end_str   = df_raw.index.max().strftime('%Y-%m-%d')
    nor_raw, nor_report = load_norway_raw(start_str, end_str, NOR_CACHE, live=False)
    print_connectivity_report(nor_report)

    if not nor_raw.empty:
        df_raw = df_raw.join(nor_raw, how='left')
        df_raw = add_norway_features(df_raw, nor_raw)
        print(f'  Norway nor_* columns added: '
              f'{sum(1 for c in df_raw.columns if c.startswith("nor_"))}')
    else:
        print('  [WARN] Norway data unavailable — nor_* features will be NaN '
              '(CatBoost handles NaN natively; nor_* columns will have no predictive content)')

    print('Adding macro features ...')
    df_raw = add_macro_features(df_raw)
    print(f'  df_raw shape after augmentation: {df_raw.shape}')
    return df_raw


df_raw = load_and_augment_data()

_SWAP_PAT = re.compile(r'^[A-Z]+_\d+[WMY]$')
swap_cols = sorted(c for c in df_raw.columns if _SWAP_PAT.match(c))
obs       = df_raw[swap_cols].notna().sum()
universe  = sorted(obs[obs >= UNIVERSE_MIN_OBS].index.tolist())
universe  = [s for s in universe if s.rsplit('_', 1)[0] == 'NOR']
print(f'\nNorwegian swap universe ({len(universe)} securities): {universe}')
assert SMOKE_SEC in universe, f'{SMOKE_SEC} not in NOR universe — check data'

### Metrics contract — shared long-CSV schema

Every results row uses `compute_metrics_row` / `SHARED_COLS` from `src.evaluation.metrics` —
the single source of truth that all three models (HTBoost, XGBoost, CatBoost) share.
Rows produced here merge with `v5_metrics_long.csv` and `xgb_metrics_long*.csv`
with zero reconciliation.

In [ ]:
# Shared evaluation harness — single source of truth; NOT redefined here.
# compute_metrics_row, SHARED_COLS, apply_mtc_family, finalize_long_csv
# are already imported at the top of this notebook.
print(f'SHARED_COLS ({len(SHARED_COLS)}): {SHARED_COLS}')

### Cross-market PCA (fit on training rows only)

`fit_transform_xm_pca` from `src.features.pca`: standardise on training moments,
zero-fill NaN, fit PCA on training rows only, apply to both train and test.
k = smallest number of components explaining ≥ `XM_PCA_VAR` of training variance,
capped at `XM_PCA_KMAX`. Identical rule to HTBoost and XGBoost.

In [ ]:
# fit_transform_xm_pca is already imported at the top.
# Confirming the shared PCA rule matches config:
print(f'PCA rule: XM_PCA_ENABLE={XM_PCA_ENABLE}  var_target={XM_PCA_VAR}  kmax={XM_PCA_KMAX}')
print('fit_transform_xm_pca imported from src.features.pca (fit on TRAIN only).')

### CatBoost tune + fit + extraction (all importance types · native SHAP · learning curve)

In [ ]:
def _tune_and_fit_cat(y_tr, x_tr_df, y_te, x_te_df, H, seed=config.JULIA_SEED):
    """Time-ordered inner-CV grid search + refit. The tuning/fit CORE is unchanged (matched-pair:
    same seed, thread_count=1, MVS default, oblivious depth, H-1 inner purge); this ADDS extraction
    of all per-feature importance types, native SHAP (ShapValues), and the learning curve.

    Documented divergences preserved: RMSE vs HTBoost Student-t; MVS data fraction (logged as
    cat_mvs_fraction, NOT Bernoulli subsample); symmetric/oblivious depth [4,6] (not == XGBoost).
    """
    n           = len(y_tr)
    n_inner_val = max(MIN_TEST_OBS, int(n * INNER_VAL_FRAC))
    n_inner_tr  = n - n_inner_val
    n_purge     = max(0, H - 1)                       # inner purge mirrors the outer H-1 purge
    n_it_purged = n_inner_tr - n_purge
    if n_it_purged >= MIN_TRAIN_OBS:
        X_it = x_tr_df.iloc[:n_it_purged].to_numpy(dtype=np.float64); y_it = y_tr[:n_it_purged]
        use_early_stopping = True
    else:
        X_it = x_tr_df.iloc[:n_inner_tr].to_numpy(dtype=np.float64);  y_it = y_tr[:n_inner_tr]
        use_early_stopping = False
    X_iv = x_tr_df.iloc[n_inner_tr:].to_numpy(dtype=np.float64); y_iv = y_tr[n_inner_tr:]
    feat_names = list(x_tr_df.columns)               # saved before numpy conversion

    keys, values = list(CAT_PARAM_GRID.keys()), list(CAT_PARAM_GRID.values())
    best_mse, best_params, best_n_est, tuning_rows = np.inf, None, N_TREES_MAX, []
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        mdl_kw = dict(iterations=N_TREES_MAX, loss_function=LOSS_FUNCTION, random_seed=seed,
                      thread_count=1, logging_level='Silent', allow_writing_files=False, **params)
        if use_early_stopping:
            mdl = CatBoostRegressor(**mdl_kw)
            mdl.fit(X_it, y_it, eval_set=(X_iv, y_iv), early_stopping_rounds=EARLY_STOP_RND)
            n_est = int(getattr(mdl, 'best_iteration_', N_TREES_MAX - 1)) + 1
        else:
            mdl = CatBoostRegressor(**mdl_kw); mdl.fit(X_it, y_it); n_est = N_TREES_MAX
        mse_iv = float(np.mean((y_iv - mdl.predict(X_iv)) ** 2))
        log_params = {('cat_mvs_fraction' if k == 'subsample' else k): v for k, v in params.items()}
        tuning_rows.append({**log_params, 'val_mse': mse_iv, 'best_n_estimators': n_est, 'selected': False})
        if mse_iv < best_mse:
            best_mse, best_params, best_n_est = mse_iv, params.copy(), n_est
    for row in tuning_rows:
        if all(row.get('cat_mvs_fraction' if k == 'subsample' else k) == v
               for k, v in best_params.items()):
            row['selected'] = True; break

    # Final model on the FULL training window (predictions)
    mdl_final = CatBoostRegressor(iterations=best_n_est, loss_function=LOSS_FUNCTION, random_seed=seed,
                                  thread_count=1, logging_level='Silent', allow_writing_files=False,
                                  **best_params)
    X_tr_np, X_te_np = x_tr_df.to_numpy(dtype=np.float64), x_te_df.to_numpy(dtype=np.float64)
    mdl_final.fit(X_tr_np, y_tr)
    yhat_tr, yhat_te = mdl_final.predict(X_tr_np), mdl_final.predict(X_te_np)

    # (#4) ALL per-feature importance types
    imp_by_type = {}
    try:
        imp_by_type['PredictionValuesChange'] = {fn: float(v) for fn, v in zip(
            feat_names, mdl_final.get_feature_importance(type='PredictionValuesChange'))}
    except Exception as e: print(f'    [imp PVC] skipped - {repr(e)[:50]}')
    try:  # LossFunctionChange needs a labelled training Pool
        imp_by_type['LossFunctionChange'] = {fn: float(v) for fn, v in zip(
            feat_names, mdl_final.get_feature_importance(Pool(X_tr_np, y_tr), type='LossFunctionChange'))}
    except Exception as e: print(f'    [imp LFC] skipped - {repr(e)[:50]}')

    # (#5) NATIVE SHAP (no shap lib needed): ShapValues -> (n, n_feat+1), last col = bias
    shap_mean_abs = {}
    try:
        sv = mdl_final.get_feature_importance(Pool(X_te_np), type='ShapValues')
        shap_mean_abs = {fn: float(v) for fn, v in zip(feat_names, np.abs(sv[:, :-1]).mean(axis=0))}
    except Exception as e: print(f'    [shap] skipped - {repr(e)[:50]}')

    # (#6) learning curve for the SELECTED config (one extra fit; CatBoost evals_result -> learn/validation)
    eval_history = []
    try:
        mc = CatBoostRegressor(iterations=best_n_est, loss_function=LOSS_FUNCTION, random_seed=seed,
                               thread_count=1, logging_level='Silent', allow_writing_files=False, **best_params)
        mc.fit(X_it, y_it, eval_set=(X_iv, y_iv))
        ev = mc.get_evals_result()
        _lr = ev.get('learn', {}).get(LOSS_FUNCTION, [])
        _va = ev.get('validation', {}).get(LOSS_FUNCTION, [])
        eval_history = [{'round': i, 'train_rmse': float(t), 'val_rmse': float(v)}
                        for i, (t, v) in enumerate(zip(_lr, _va))]
    except Exception as e: print(f'    [eval_history] skipped - {repr(e)[:50]}')

    return dict(yhat_tr=yhat_tr, yhat_te=yhat_te, best_params=best_params, best_n_est=best_n_est,
                tuning_rows=tuning_rows, imp_by_type=imp_by_type, shap_mean_abs=shap_mean_abs,
                eval_history=eval_history)
print('_tune_and_fit_cat defined (tuning/fit core unchanged; +all-imp +native SHAP +eval_history)')

### Shared fold scorer + walk-forward / block-CV·LORO runners

In [ ]:
def _fit_score_fold(tr, te, fc, H, sec, country, tenor, scheme, fold_name, regime, seed=config.JULIA_SEED):
    """PCA-compress (train only) -> tune+fit -> SHARED_COLS train+oos rows + preds + imps + tuning + curve.
    Identical fit path for WF and block-CV/LORO; only the train/test masks and `scheme` differ."""
    x_tr, x_te, pca = fit_transform_xm_pca(tr[fc], te[fc])
    fcount = x_tr.shape[1]
    x_tr_c, x_te_c = _prepare_x_cat(x_tr), _prepare_x_cat(x_te)
    y_tr, y_te = tr['y'].to_numpy(float), te['y'].to_numpy(float)
    r = _tune_and_fit_cat(y_tr, x_tr_c, y_te, x_te_c, H, seed=seed)
    cfg = {**r['best_params'], 'n_trees': r['best_n_est'], 'loss_function': LOSS_FUNCTION,
           'inner_val_frac': INNER_VAL_FRAC}
    meta = dict(notebook=NOTEBOOK, run_ts=RUN_TS, model_kind=MODEL_KIND, is_pooled=IS_POOLED,
                validation_scheme=scheme, target_kind='level_change', security=sec, country=country,
                tenor=tenor, fold=fold_name, regime=regime,
                config_hash=_config_hash(cfg, extra=FEAT_SPEC), feature_count=fcount)
    rows = []
    for samp, yt, yh in (('train', y_tr, r['yhat_tr']), ('oos', y_te, r['yhat_te'])):
        row = compute_metrics_row(yt, yh, H, {**meta, 'sample': samp})
        row['pca_k'] = pca['k']; row['xm_pca_evr'] = pca['evr_cum_k']
        row['cat_n_trees'] = r['best_n_est']
        row['cat_depth'] = r['best_params'].get('depth', -1)
        row['cat_lr'] = r['best_params'].get('learning_rate', -1)
        rows.append(row)
    preds = pd.DataFrame({'date': te['date'].values, 'security': sec, 'tenor': tenor,
                          'horizon': int(H), 'regime': regime, 'scheme': scheme,
                          'y_true': y_te, 'y_pred': np.asarray(r['yhat_te'], float)})
    imp_rec = {'security': sec, 'horizon': H, 'fold': fold_name, 'regime': regime, 'scheme': scheme,
               'imp_by_type': r['imp_by_type'], 'shap': r['shap_mean_abs']}
    for trow in r['tuning_rows']:
        trow.update({'security': sec, 'horizon': H, 'fold': fold_name, 'regime': regime, 'scheme': scheme})
    evh = [{'security': sec, 'horizon': H, 'fold': fold_name, 'scheme': scheme, **e} for e in r['eval_history']]
    return rows, imp_rec, preds, r['tuning_rows'], evh

def _run_security_wf(sec, df_raw, H, seed=config.JULIA_SEED):
    if sec not in df_raw.columns: return [], [], [], [], []
    panel = build_panel(df_raw, [sec], H)
    if len(panel) == 0: return [], [], [], [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    country, tenor = sec.rsplit('_', 1)
    rows, imps, preds, tun, evh = [], [], [], [], []
    for fold_name, ts, te_end, regime in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te_end)
        purge_ts = ts_ts - pd.tseries.offsets.BDay(H - 1)             # one-sided H-1 purge
        trn = panel[panel['date'] < purge_ts].copy()
        tst = panel[(panel['date'] >= ts_ts) & (panel['date'] <= te_ts)].copy()
        if len(trn) < MIN_TRAIN_OBS or len(tst) < MIN_TEST_OBS: continue
        r, ir, pr, tr_rows, eh = _fit_score_fold(trn, tst, fc, H, sec, country, tenor,
                                                 'walk_forward', fold_name, regime, seed)
        rows += r; imps.append(ir); preds.append(pr); tun += tr_rows; evh += eh
    return rows, imps, preds, tun, evh

def _blockcv_entries(scheme):
    if scheme == 'block_cv':
        return [(name, regime, [(pd.Timestamp(s), pd.Timestamp(e))]) for (name, s, e, regime) in BLOCK_CV_FOLDS]
    if scheme == 'loro':
        out = []
        for regime in dict.fromkeys(r for (_n, _s, _e, r) in BLOCK_CV_FOLDS):
            segs = [(pd.Timestamp(s), pd.Timestamp(e)) for (_n, s, e, r) in BLOCK_CV_FOLDS if r == regime]
            out.append((f'LORO_{regime}', regime, segs))
        return out
    raise ValueError(scheme)

def _run_security_blockcv(sec, df_raw, scheme, H, seed=config.JULIA_SEED):
    """Non-causal learnability diagnostic: two-sided purge+embargo around each held-out block."""
    if sec not in df_raw.columns: return [], [], [], [], []
    panel = build_panel(df_raw, [sec], H)
    if len(panel) == 0: return [], [], [], [], []
    fc = [c for c in panel.columns if c not in META_COLS]
    country, tenor = sec.rsplit('_', 1)
    gap = (H - 1) + EMBARGO_FOR_H(H)
    dates = panel['date']
    rows, imps, preds, tun, evh = [], [], [], [], []
    for label, regime, segs in _blockcv_entries(scheme):
        test_mask = pd.Series(False, index=panel.index)
        for (s, e) in segs: test_mask |= (dates >= s) & (dates <= e)
        train_mask = ~test_mask
        for (s, e) in segs:                                          # two-sided symmetric gap
            train_mask &= ~((dates >= s - pd.tseries.offsets.BDay(gap)) & (dates <= e + pd.tseries.offsets.BDay(gap)))
        trn, tst = panel[train_mask].copy(), panel[test_mask].copy()
        if len(trn) < MIN_TRAIN_OBS or len(tst) < MIN_TEST_OBS: continue
        r, ir, pr, tr_rows, eh = _fit_score_fold(trn, tst, fc, H, sec, country, tenor,
                                                 scheme, label, regime, seed)
        rows += r; imps.append(ir); preds.append(pr); tun += tr_rows; evh += eh
    return rows, imps, preds, tun, evh

def _imps_long(imp_recs):
    out = []
    for r in imp_recs:
        for t, d in r['imp_by_type'].items():
            for feat, val in d.items():
                out.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                            'regime': r['regime'], 'scheme': r['scheme'], 'feature': feat,
                            'importance_type': t, 'value': val, 'bucket': bucket_feature(feat)})
        for feat, val in (r.get('shap') or {}).items():
            out.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                        'regime': r['regime'], 'scheme': r['scheme'], 'feature': feat,
                        'importance_type': 'shap_mean_abs', 'value': val, 'bucket': bucket_feature(feat)})
    return pd.DataFrame(out)

def _shap_buckets(imp_recs):
    rows = []
    for r in imp_recs:
        agg = {b: 0.0 for b in BUCKETS}
        for feat, val in (r.get('shap') or {}).items():
            agg[bucket_feature(feat)] += abs(float(val))
        tot = sum(agg.values()) or 1.0
        for b in BUCKETS:
            rows.append({'security': r['security'], 'horizon': r['horizon'], 'fold': r['fold'],
                         'scheme': r['scheme'], 'bucket': b, 'shap_share_pct': 100.0 * agg[b] / tot})
    return pd.DataFrame(rows)
print('runners defined: WF + block_cv/loro (shared _fit_score_fold); _imps_long, _shap_buckets')

### Horizon support gate

In [ ]:
def _horizon_supported(df_raw, h):
    panel = build_panel(df_raw, [SMOKE_SEC], h)
    if len(panel) == 0: return False
    for _, ts, te, _ in WF_FOLDS:
        ts_ts, te_ts = pd.Timestamp(ts), pd.Timestamp(te)
        purge = ts_ts - pd.tseries.offsets.BDay(h - 1)
        if (panel['date'] < purge).sum() >= MIN_TRAIN_OBS and \
           ((panel['date'] >= ts_ts) & (panel['date'] <= te_ts)).sum() >= MIN_TEST_OBS:
            return True
    return False

HORIZONS = list(H_GRID) + [h for h in H_GRID_LONG if _horizon_supported(df_raw, h)]
HORIZONS = sorted(set(HORIZONS))
print('Final horizon grid:', HORIZONS)

### ⭐ ONE-CELL DATA-MACHINE SMOKE (run this single cell)

Floor-scale integration check for a machine that has the Bloomberg data + `catboost`.
**Pre-reqs:** `export THESIS_DATA_PATH=/path/to/Data`; `catboost==1.2.10` installed (pinned in
`environment.yml`). Native SHAP needs no `shap` lib; the swarm figure is hand-rolled (no dep).
Run the definition cells above, then **this one cell**: it sets all sweep/push switches OFF and
exercises the full machinery once: load → features → PCA → tune+fit → all importance types →
SHAP → eval_history → **one block-CV fold** → the three figures → **six** files in
`results/catboost_v5_nor/_smoke/`. Asserts on failure.

In [ ]:
# ── ONE-CELL DATA-MACHINE SMOKE — additive; uses only functions defined above ──
RUN_WF = RUN_BLOCKCV = RUN_LORO = RUN_PUSH = False     # belt-and-suspenders: no sweep, no push
os.makedirs(SMOKE_DIR, exist_ok=True)
_SFIG = os.path.join(SMOKE_DIR, 'figures'); os.makedirs(_SFIG, exist_ok=True)
if 'df_raw' not in dir():
    df_raw = load_and_augment_data()
print(f'[SMOKE] {SMOKE_SEC} x H={SMOKE_H}')

# (1) walk-forward smoke (all WF folds, one tenor/horizon)
wf_rows, wf_imps, wf_preds, wf_tun, wf_evh = _run_security_wf(SMOKE_SEC, df_raw, SMOKE_H)
assert wf_rows, 'WF smoke produced no rows'

# (2) ONE non-causal block-CV fold (first block) via the shared scorer — floor-scale path check
_panel = build_panel(df_raw, [SMOKE_SEC], SMOKE_H)
_fc = [c for c in _panel.columns if c not in META_COLS]
_country, _tenor = SMOKE_SEC.rsplit('_', 1)
_gap = (SMOKE_H - 1) + EMBARGO_FOR_H(SMOKE_H)
_label, _regime, _segs = _blockcv_entries('block_cv')[0]
_d = _panel['date']; _tm = pd.Series(False, index=_panel.index)
for _s, _e in _segs: _tm |= (_d >= _s) & (_d <= _e)
_trm = ~_tm
for _s, _e in _segs:
    _trm &= ~((_d >= _s - pd.tseries.offsets.BDay(_gap)) & (_d <= _e + pd.tseries.offsets.BDay(_gap)))
bk_rows, bk_imp, bk_pred, bk_tun, bk_evh = _fit_score_fold(
    _panel[_trm].copy(), _panel[_tm].copy(), _fc, SMOKE_H, SMOKE_SEC, _country, _tenor,
    'block_cv', _label, _regime)
assert bk_rows, 'block-CV smoke produced no rows'

# (3) write the SIX WF file types + the block-CV files to _smoke/
pd.DataFrame(wf_rows).to_csv(_wf_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.concat(wf_preds, ignore_index=True).to_csv(_pred_csv(SMOKE_H, SMOKE_DIR), index=False)
_imps_long(wf_imps).to_csv(_imp_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.DataFrame(wf_tun).to_csv(_tuning_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.DataFrame(wf_evh).to_csv(_evalhist_csv(SMOKE_H, SMOKE_DIR), index=False)
_shap_buckets(wf_imps).to_csv(_shap_csv(SMOKE_H, SMOKE_DIR), index=False)
pd.DataFrame(bk_rows).to_csv(_blk_csv(SMOKE_H, SMOKE_DIR), index=False)
bk_pred.to_csv(_blk_pred_csv(SMOKE_H, SMOKE_DIR), index=False)
_imps_long([bk_imp]).to_csv(_blk_imp_csv(SMOKE_H, SMOKE_DIR), index=False)

# (4) integration assertions the unit tests could not prove
_il = _imps_long(wf_imps)
_need = {'PredictionValuesChange', 'LossFunctionChange', 'shap_mean_abs'}
assert _need.issubset(set(_il['importance_type'])), f'missing types: {_need - set(_il["importance_type"])}'
_oos = pd.DataFrame(wf_rows).query("sample == 'oos'")
_extra = ['pca_k', 'xm_pca_evr', 'cat_n_trees', 'cat_depth', 'cat_lr']
assert set(SHARED_COLS).issubset(_oos.columns), 'metrics row missing SHARED_COLS'
assert set(_extra).issubset(_oos.columns), 'metrics row missing cat extras'

# (5) the three figures -> _smoke/figures/
_g = _il[_il.importance_type == 'PredictionValuesChange'].groupby('bucket')['value'].sum()
_g = (_g / _g.sum() * 100).sort_values()
_f, _ax = plt.subplots(figsize=(6.4, 3.6)); _ax.barh(_g.index, _g.values, color=OKABE_ITO[2])
_ax.set_xlabel('PredictionValuesChange share (%)'); _ax.set_title(f'CatBoost importance by bucket - {SMOKE_SEC} H={SMOKE_H}')
_f.tight_layout(); _f.savefig(os.path.join(_SFIG, 'cat_smoke_importance_buckets.pdf')); plt.close(_f)

# SHAP swarm (native ShapValues; hand-rolled beeswarm so NO shap lib is needed)
_ts = pd.Timestamp(WF_FOLDS[-1][1]); _te_end = pd.Timestamp(WF_FOLDS[-1][2])
_trn = _panel[_panel['date'] < _ts - pd.tseries.offsets.BDay(SMOKE_H - 1)]
_tst = _panel[(_panel['date'] >= _ts) & (_panel['date'] <= _te_end)]
_xtr, _xte, _ = fit_transform_xm_pca(_trn[_fc], _tst[_fc]); _xtr, _xte = _prepare_x_cat(_xtr), _prepare_x_cat(_xte)
_m = CatBoostRegressor(iterations=200, loss_function=LOSS_FUNCTION, depth=4, learning_rate=0.05,
                       random_seed=config.JULIA_SEED, thread_count=1, logging_level='Silent',
                       allow_writing_files=False).fit(_xtr.to_numpy(np.float64), _trn['y'].to_numpy(float))
_sv = _m.get_feature_importance(Pool(_xte.to_numpy(np.float64)), type='ShapValues')[:, :-1]
_cols = list(_xte.columns); _order = np.argsort(np.abs(_sv).mean(0))[::-1][:15]
_f, _ax = plt.subplots(figsize=(7.0, 4.6))
for _row, _j in enumerate(_order):
    _vals = _sv[:, _j]
    _fv = _xte.iloc[:, _j].to_numpy(); _rng = np.nanmax(_fv) - np.nanmin(_fv)
    _cv = (_fv - np.nanmin(_fv)) / (_rng + 1e-9) if _rng > 0 else np.full_like(_fv, 0.5)
    _y = _row + (np.random.default_rng(_j).random(len(_vals)) - 0.5) * 0.6   # jitter
    _sc = _ax.scatter(_vals, _y, c=_cv, cmap='coolwarm', s=6, alpha=0.6, linewidths=0)
_ax.set_yticks(range(len(_order))); _ax.set_yticklabels([_cols[j] for j in _order], fontsize=7)
_ax.invert_yaxis(); _ax.axvline(0, color='#999', lw=0.8)
_ax.set_xlabel('SHAP value (impact on prediction)'); _ax.set_title(f'CatBoost SHAP swarm - {SMOKE_SEC} H={SMOKE_H}')
_cb = _f.colorbar(_sc, ax=_ax, fraction=0.025, pad=0.02); _cb.set_label('feature value (low->high)', fontsize=7)
_f.tight_layout(); _f.savefig(os.path.join(_SFIG, 'cat_smoke_shap_swarm.pdf'), bbox_inches='tight'); plt.close(_f)

_eh = pd.DataFrame(wf_evh)
if len(_eh):
    _k = _eh[['security', 'horizon', 'fold']].drop_duplicates().iloc[0]
    _s = _eh[(_eh.security == _k.security) & (_eh.horizon == _k.horizon) & (_eh.fold == _k.fold)].sort_values('round')
    _f, _ax = plt.subplots(figsize=(6.4, 3.6))
    _ax.plot(_s['round'], _s['train_rmse'], label='train', color=OKABE_ITO[0])
    _ax.plot(_s['round'], _s['val_rmse'], label='inner-val', color=OKABE_ITO[3])
    _ax.set_xlabel('boosting round'); _ax.set_ylabel('RMSE'); _ax.legend(frameon=False)
    _ax.set_title(f'CatBoost early-stopping - {_k.security} H={_k.horizon} {_k.fold}')
    _f.tight_layout(); _f.savefig(os.path.join(_SFIG, 'cat_smoke_learning_curve.pdf')); plt.close(_f)

# (6) confirm the SIX WF files exist, then summarize
_six = [_wf_csv(SMOKE_H, SMOKE_DIR), _pred_csv(SMOKE_H, SMOKE_DIR), _imp_csv(SMOKE_H, SMOKE_DIR),
        _tuning_csv(SMOKE_H, SMOKE_DIR), _evalhist_csv(SMOKE_H, SMOKE_DIR), _shap_csv(SMOKE_H, SMOKE_DIR)]
_missing = [os.path.basename(p) for p in _six if not os.path.exists(p)]
assert not _missing, f'missing smoke files: {_missing}'
print(f'  WF OOS folds={len(_oos)}  block-CV OOS rows={sum(1 for r in bk_rows if r["sample"]=="oos")}')
print(f'  importance types: {sorted(_need & set(_il["importance_type"]))}')
print(f'  SIX _smoke files: {[os.path.basename(p) for p in _six]}')
print(f'  figures: ["cat_smoke_importance_buckets.pdf", "cat_smoke_shap_swarm.pdf", "cat_smoke_learning_curve.pdf"]')
print('[SMOKE PASS] load -> features -> PCA -> tune+fit -> all-imp -> native SHAP -> eval_history '
      '-> block-CV fold -> 3 figures -> 6 files OK')

### Walk-forward sweep (per-horizon checkpoint + auto-push)

In [ ]:
if RUN_WF:
    for H_run in HORIZONS:
        wf_csv_p  = _wf_csv(H_run, OUT_DIR)
        imp_pkl_p = wf_csv_p.replace('.csv', '_imps.pkl')
        done      = _done_secs(wf_csv_p)
        all_imps  = _load_pkl(imp_pkl_p)
        if done: print(f'  [H={H_run}] resume: {len(done)} security(ies) already on disk')
        for sec in universe:
            if sec in done: continue
            t0 = time.time()
            try:
                rows, imps, preds, tun, evh = _run_security_wf(sec, df_raw, H_run)
            except Exception as e:
                print(f'  [H={H_run}] {sec}: FAILED {repr(e)[:70]}'); continue
            if rows:  _append_csv(pd.DataFrame(rows), wf_csv_p)
            if preds: _append_csv(pd.concat(preds, ignore_index=True), _pred_csv(H_run, OUT_DIR))
            if tun:   _append_csv(pd.DataFrame(tun), _tuning_csv(H_run, OUT_DIR))
            if evh:   _append_csv(pd.DataFrame(evh), _evalhist_csv(H_run, OUT_DIR))
            all_imps += imps
            _dump_pkl(imp_pkl_p, all_imps)
            _imps_long(all_imps).to_csv(_imp_csv(H_run, OUT_DIR), index=False)      # full rewrite from accum
            _shap_buckets(all_imps).to_csv(_shap_csv(H_run, OUT_DIR), index=False)
            oos = [r for r in rows if r.get('sample') == 'oos']
            print(f'  [H={H_run}] {sec:<10s} folds={len(oos)} ({time.time()-t0:.1f}s)')
        _maybe_push([_wf_csv(H_run, OUT_DIR), _pred_csv(H_run, OUT_DIR), _imp_csv(H_run, OUT_DIR),
                     _tuning_csv(H_run, OUT_DIR), _evalhist_csv(H_run, OUT_DIR), _shap_csv(H_run, OUT_DIR)],
                    f'results: catboost_v5_nor walk_forward H{H_run} @ {MACHINE_ID}')
    print('WF sweep done')
else:
    print('[skipped] RUN_WF=False')

### Block-CV / LORO sweep (non-causal learnability family + auto-push)

In [ ]:
_schemes = (['block_cv'] if RUN_BLOCKCV else []) + (['loro'] if RUN_LORO else [])
if _schemes:
    for H_run in HORIZONS:
        blk_csv_p = _blk_csv(H_run, OUT_DIR)
        imp_pkl_p = blk_csv_p.replace('.csv', '_imps.pkl')
        done_keys = set()
        if os.path.exists(blk_csv_p):
            _d = pd.read_csv(blk_csv_p, usecols=['security', 'validation_scheme'])
            done_keys = set(_d['security'] + '|' + _d['validation_scheme'])
        blk_imps = _load_pkl(imp_pkl_p)
        for scheme in _schemes:
            for sec in universe:
                if f'{sec}|{scheme}' in done_keys: continue
                t0 = time.time()
                try:
                    rows, imps, preds, tun, evh = _run_security_blockcv(sec, df_raw, scheme, H_run)
                except Exception as e:
                    print(f'  [H={H_run}] {scheme} {sec}: FAILED {repr(e)[:60]}'); continue
                if rows:  _append_csv(pd.DataFrame(rows), blk_csv_p)
                if preds: _append_csv(pd.concat(preds, ignore_index=True), _blk_pred_csv(H_run, OUT_DIR))
                if tun:   _append_csv(pd.DataFrame(tun), _tuning_csv(H_run, OUT_DIR))
                blk_imps += imps
                _dump_pkl(imp_pkl_p, blk_imps)
                _imps_long(blk_imps).to_csv(_blk_imp_csv(H_run, OUT_DIR), index=False)
                print(f'  [H={H_run}] {scheme:<9s} {sec:<10s} ({time.time()-t0:.1f}s)')
        _maybe_push([_blk_csv(H_run, OUT_DIR), _blk_pred_csv(H_run, OUT_DIR), _blk_imp_csv(H_run, OUT_DIR)],
                    f'results: catboost_v5_nor block_cv+loro H{H_run} @ {MACHINE_ID}')
    print('block-CV/LORO sweep done')
else:
    print('[skipped] RUN_BLOCKCV=RUN_LORO=False')

### Assemble metrics + MTC (SEPARATE walk-forward and learnability families)

In [ ]:
def _read_all(globpat):
    import glob
    parts = [pd.read_csv(p) for p in sorted(glob.glob(globpat)) if os.path.getsize(p) > 0]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

wf  = _read_all(_wf_csv('*', OUT_DIR))
blk = _read_all(_blk_csv('*', OUT_DIR))
df_all = finalize_long_csv(pd.concat([d for d in (wf, blk) if len(d)], ignore_index=True)) \
         if (len(wf) or len(blk)) else pd.DataFrame(columns=SHARED_COLS)
if len(df_all):
    # Causal forecastability family (walk_forward), corrected within this estimator
    apply_mtc_family(df_all, ['walk_forward'], 'walk_forward:{horizon×tenor×regime}', model_kind=MODEL_KIND)
    # Non-causal learnability family (block_cv + loro) — kept SEPARATE, never pooled with WF
    apply_mtc_family(df_all, ['block_cv', 'loro'], 'block_cv+loro:{horizon×tenor×regime}', model_kind=MODEL_KIND)
    df_all.to_csv(os.path.join(OUT_DIR, f'cat_metrics_long__{MACHINE_ID}.csv'), index=False)
    print(f'metrics_long rows={len(df_all)} cols={df_all.shape[1]}')
    print(df_all.groupby('validation_scheme')['security'].count())
else:
    print('no metrics yet (run the sweeps)')

### Figures — importance buckets · learning curve

In [ ]:
_imp = _read_all(_imp_csv('*', OUT_DIR))
_eh  = _read_all(_evalhist_csv('*', OUT_DIR))
if not len(_imp):
    _imp = _read_all(_imp_csv('*', SMOKE_DIR)); _eh = _read_all(_evalhist_csv('*', SMOKE_DIR))
if len(_imp):
    g = _imp[_imp['importance_type'] == 'PredictionValuesChange'].groupby('bucket')['value'].sum()
    g = (g / g.sum() * 100).sort_values()
    fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.barh(g.index, g.values, color=OKABE_ITO[2])
    ax.set_xlabel('PredictionValuesChange share (%)'); ax.set_title('CatBoost importance by bucket')
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'cat_importance_buckets.pdf')); plt.close(fig)
    print('saved cat_importance_buckets.pdf')
if len(_eh):
    one = _eh.sort_values('round'); k = one[['security','horizon','fold']].drop_duplicates().iloc[0]
    sub = one[(one.security==k.security)&(one.horizon==k.horizon)&(one.fold==k.fold)]
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    ax.plot(sub['round'], sub['train_rmse'], label='train', color=OKABE_ITO[0])
    ax.plot(sub['round'], sub['val_rmse'], label='inner-val', color=OKABE_ITO[3])
    ax.set_xlabel('boosting round'); ax.set_ylabel('RMSE'); ax.legend(frameon=False)
    ax.set_title(f'CatBoost early-stopping - {k.security} H={k.horizon} {k.fold}')
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'cat_learning_curve.pdf')); plt.close(fig)
    print('saved cat_learning_curve.pdf')